# Clase 3 · Laboratorio 1
## Manejo de clusters y notebooks colaborativos para Ciencia de Datos a escala

**DevSeniorCode — Máster en Inteligencia Artificial & Data Science**
Módulo 3 · Unidad 3 · Clase 3

---

### Objetivo del laboratorio
1. Verificar que el notebook está conectado a un **cluster** activo y explorar su configuración.
2. Repasar los **magic commands** más usados en Databricks (`%md`, `%sql`, `%fs`, `%sh`, `%run`, `%pip`).
3. Leer un dataset con **PySpark** y ejecutar transformaciones distribuidas (`select`, `filter`, `groupBy`, `agg`, `join`).
4. Comparar una operación equivalente en **Pandas vs. PySpark**.
5. Guardar el resultado como una tabla para que el equipo la reutilice (trabajo colaborativo).

> **Cómo usar este notebook:** impórtalo a tu Workspace de **Databricks Free Edition** (regístrate gratis en docs.databricks.com/getting-started/free-edition), conéctalo a **Serverless** (Databricks lo aprovisiona automáticamente, sin crear un cluster manual) y ejecuta las celdas en orden con `Shift + Enter`.
>
> Nota: Free Edition es 100% serverless — los conceptos de "cluster" que vemos en la teoría (all-purpose, job cluster, autoscaling) aplican a los workspaces con cómputo clásico (cuentas de pago); aun así son fundamentales para entender cómo funciona Databricks por dentro.


## 1. Explorando el cómputo desde el notebook

Cuando un notebook está **conectado** a un cluster o a **Serverless**, Databricks ya te entrega la `SparkSession` lista para usar en la variable `spark`.

> ⚠️ **Importante en Serverless (Databricks Free Edition):** el objeto clásico `sc` (`SparkContext`) **no está disponible** — Databricks lo abstrae por completo (`SparkContext is not supported on serverless compute`). Esto es intencional: en serverless no gestionas nodos ni configuración de bajo nivel, solo escribes DataFrames y Databricks decide cómo distribuir el trabajo. `sc` sigue existiendo en clusters clásicos (cómputo dedicado / cuentas de pago).

Vamos a inspeccionar lo que sí está disponible en cualquier modo: la versión de Spark y algunos parámetros de configuración expuestos vía `spark.conf`.

In [ ]:
# Si este notebook corre FUERA de Databricks (ej. Jupyter local), creamos
# una SparkSession local para que el resto del notebook funcione igual.
try:
    spark  # ya existe en Databricks (cluster clásico o Serverless)
    print("Conectado al cómputo de Databricks ✅")
except NameError:
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .appName("Clase3_Clusters_Notebooks")
        .master("local[*]")
        .config("spark.driver.host", "127.0.0.1")
        .getOrCreate()
    )
    print("SparkSession local creada (modo demo fuera de Databricks) ⚠️")

print("Versión de Spark:", spark.version)

# sc (SparkContext) solo existe en cómputo clásico, NO en Serverless.
try:
    sc = spark.sparkContext
    print("Master:", sc.master)
    print("Paralelismo por defecto (núcleos disponibles):", sc.defaultParallelism)
except Exception as e:
    sc = None
    print("sc (SparkContext) no está disponible en este cómputo (típico en Serverless).")
    print("Detalle:", type(e).__name__)


In [ ]:
# En Serverless, la forma soportada de leer configuración es spark.conf.get(),
# NO sc.getConf() (que depende del SparkContext, no disponible en serverless).
keys_interes = ["spark.app.name", "spark.sql.shuffle.partitions"]
for k in keys_interes:
    try:
        print(f"{k} = {spark.conf.get(k)}")
    except Exception:
        print(f"{k} = (no expuesto en este modo de cómputo)")


### `dbutils`: la caja de herramientas del Workspace

`dbutils` **solo existe dentro de un notebook de Databricks** (no en Jupyter local). Permite:

- `dbutils.fs` → operar sobre DBFS (equivalente al magic command `%fs`).
- `dbutils.widgets` → crear parámetros interactivos (dropdowns, text boxes) para el notebook.
- `dbutils.notebook.run()` → encadenar la ejecución de otros notebooks.
- `dbutils.secrets` → leer credenciales sin exponerlas en el código.

Ejecuta la siguiente celda **solo si estás en Databricks**.

In [ ]:
try:
    # Listamos el contenido raíz de DBFS
    display(dbutils.fs.ls("/databricks-datasets"))
except NameError:
    print("dbutils no está disponible fuera de Databricks — esta celda es solo demostrativa.")


## 2. Magic Commands — referencia rápida

En un notebook real de Databricks, cada celda puede cambiar de lenguaje con un *magic command* en la primera línea:

| Comando | Uso |
|---|---|
| `%python` | Código Python (lenguaje por defecto en este notebook) |
| `%sql` | Consultas SQL directas sobre tablas del catálogo |
| `%scala` | Código Scala en la misma sesión de Spark |
| `%md` | Texto en Markdown, como esta celda |
| `%fs` | Comandos sobre el sistema de archivos DBFS (`ls`, `cp`, `rm`, `mkdirs`) |
| `%sh` | Comandos de shell dentro del nodo Driver |
| `%run "/ruta/otro_notebook"` | Ejecuta otro notebook como si fuera un módulo importado |
| `%pip install <paquete>` | Instala una librería directamente en el cluster |

> ⚠️ En los notebooks **serverless** de Databricks Free Edition, `%scala` y `%r` no están disponibles — solo Python y SQL. En workspaces con cómputo clásico (cuentas de pago) sí puedes usarlos.

**Ejemplo (copiar en una celda nueva dentro de Databricks):**

```
%fs ls /databricks-datasets/samples

%sql
SELECT current_date(), current_user()
```


## 3. Cargando datos a escala con PySpark

Vamos a crear un dataset sintético de **ventas** (como si viniera de un sistema transaccional) y lo leeremos como lo haríamos con un archivo real en producción: con `spark.read`. En un escenario real, esta misma línea leería miles de archivos Parquet/CSV distribuidos en el Data Lake sin cambiar una sola línea de código.

In [ ]:
import random
from datetime import date, timedelta

random.seed(42)
tiendas = ["Bogota", "Medellin", "Cali", "Barranquilla", "Bucaramanga"]
categorias = ["Electronica", "Hogar", "Moda", "Deportes", "Juguetes"]

filas = []
start = date(2025, 1, 1)
for i in range(5000):
    filas.append((
        i,
        random.choice(tiendas),
        random.choice(categorias),
        round(random.uniform(10, 500), 2),
        random.randint(1, 5),
        (start + timedelta(days=random.randint(0, 210))).isoformat(),
    ))

columnas = ["id_venta", "tienda", "categoria", "precio_unitario", "cantidad", "fecha"]
df_pandas_source = None  # (no usamos pandas aquí a propósito: simulamos ingesta distribuida)

df = spark.createDataFrame(filas, columnas)
print(f"Filas cargadas: {df.count():,}")
df.printSchema()
df.show(5)


## 4. Transformaciones distribuidas: `select`, `filter`, `groupBy`, `agg`

Estas operaciones son **perezosas (lazy)**: Spark no procesa nada hasta que llamamos una *acción* como `show()`, `count()` o `collect()`. Esto le permite optimizar todo el plan de ejecución antes de tocar los datos.

In [ ]:
from pyspark.sql import functions as F

# select + columna calculada
df_ventas = df.select(
    "id_venta", "tienda", "categoria", "fecha",
    (F.col("precio_unitario") * F.col("cantidad")).alias("total_venta")
)
df_ventas.show(5)


In [ ]:
# filter: solo ventas de Bogota superiores a 200
df_bogota = df_ventas.filter((F.col("tienda") == "Bogota") & (F.col("total_venta") > 200))
print("Ventas filtradas:", df_bogota.count())
df_bogota.orderBy(F.col("total_venta").desc()).show(5)


In [ ]:
# groupBy + agg: ventas totales y ticket promedio por tienda y categoria
resumen = (
    df_ventas.groupBy("tienda", "categoria")
    .agg(
        F.sum("total_venta").alias("venta_total"),
        F.avg("total_venta").alias("ticket_promedio"),
        F.count("id_venta").alias("num_transacciones"),
    )
    .orderBy(F.col("venta_total").desc())
)
resumen.show(10)


## 5. Pandas vs. PySpark: la misma operación, dos escalas distintas

Con datasets pequeños, Pandas es perfecto (todo corre en un solo núcleo, en memoria). Cuando los datos crecen más allá de lo que cabe en una máquina, necesitamos que **varios workers** trabajen en paralelo — ahí es donde PySpark demuestra su valor.

In [ ]:
import pandas as pd
import time

# Traemos una muestra a Pandas SOLO para comparar sintaxis (en un caso real, evita
# toPandas() sobre datasets grandes: se cargarían enteros en la memoria del Driver)
pdf = df_ventas.limit(2000).toPandas()

t0 = time.time()
resumen_pandas = (
    pdf.groupby(["tienda", "categoria"])
    .agg(venta_total=("total_venta", "sum"), ticket_promedio=("total_venta", "mean"))
    .reset_index()
    .sort_values("venta_total", ascending=False)
)
t_pandas = time.time() - t0

t0 = time.time()
resumen_spark = (
    df_ventas.limit(2000).groupBy("tienda", "categoria")
    .agg(F.sum("total_venta").alias("venta_total"), F.avg("total_venta").alias("ticket_promedio"))
    .orderBy(F.col("venta_total").desc())
    .toPandas()
)
t_spark = time.time() - t0

print(f"Pandas (1 núcleo):        {t_pandas*1000:.1f} ms")
print(f"PySpark (paralelo, local): {t_spark*1000:.1f} ms")
print("\nCon datasets pequeños Pandas suele ganar por el overhead de distribuir tareas.")
print("La ventaja de Spark aparece con datasets de GBs/TBs que no caben en una sola máquina.")


## 6. Joins entre DataFrames

Simulemos una tabla de **metadatos de tienda** (región, gerente) y crucémosla con nuestras ventas — un patrón habitual antes de cualquier análisis o entrenamiento de modelo.

In [ ]:
tiendas_meta = spark.createDataFrame([
    ("Bogota", "Centro", "Ana Martinez"),
    ("Medellin", "Occidente", "Carlos Ruiz"),
    ("Cali", "Occidente", "Laura Gomez"),
    ("Barranquilla", "Caribe", "Jorge Diaz"),
    ("Bucaramanga", "Oriente", "Maria Torres"),
], ["tienda", "region", "gerente"])

ventas_enriquecidas = df_ventas.join(tiendas_meta, on="tienda", how="left")
ventas_enriquecidas.groupBy("region").agg(F.sum("total_venta").alias("venta_total")).orderBy(F.desc("venta_total")).show()


## 7. Widgets: parámetros interactivos para trabajo colaborativo

Los *widgets* permiten que cualquier compañero del equipo re-ejecute tu notebook cambiando un parámetro desde la interfaz, sin tocar el código. Muy útil cuando varias personas comparten el mismo notebook (ej. elegir la tienda a analizar).

In [ ]:
try:
    dbutils.widgets.dropdown("tienda_widget", "Bogota", tiendas, "Selecciona una tienda")
    tienda_seleccionada = dbutils.widgets.get("tienda_widget")
    print("Tienda seleccionada desde el widget:", tienda_seleccionada)
    df_ventas.filter(F.col("tienda") == tienda_seleccionada).show(5)
except NameError:
    tienda_seleccionada = "Bogota"
    print("dbutils.widgets no está disponible fuera de Databricks — usando valor por defecto:", tienda_seleccionada)


## 8. Guardar resultados para el equipo (trabajo colaborativo)

En Databricks, lo habitual es guardar los resultados como una **tabla Delta** dentro del catálogo (Unity Catalog o Hive Metastore), para que cualquier compañero pueda consultarla desde SQL, otro notebook o un dashboard — sin tener que recalcular nada.

In [ ]:
try:
    (
        resumen.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("clase3_resumen_ventas")
    )
    print("Tabla 'clase3_resumen_ventas' guardada correctamente ✅")
    spark.sql("SELECT * FROM clase3_resumen_ventas LIMIT 5").show()
except Exception as e:
    print("Guardado como tabla Delta solo funciona con soporte de Delta Lake / catálogo activo.")
    print("Alternativa local: guardamos como Parquet.")
    resumen.write.mode("overwrite").parquet("/tmp/clase3_resumen_ventas_parquet")
    print("Guardado en /tmp/clase3_resumen_ventas_parquet ✅")


## 9. Consulta con SQL (equivalente al magic command `%sql`)

Una vez que el DataFrame está registrado como **vista temporal**, cualquier compañero puede consultarlo con SQL puro — ideal para analistas que no programan en Python.

In [ ]:
df_ventas.createOrReplaceTempView("ventas_temp")

resultado_sql = spark.sql('''
    SELECT tienda, categoria, ROUND(SUM(total_venta), 2) AS venta_total
    FROM ventas_temp
    GROUP BY tienda, categoria
    ORDER BY venta_total DESC
    LIMIT 10
''')
resultado_sql.show()

# Dentro de un notebook de Databricks esto sería simplemente:
# %sql
# SELECT tienda, categoria, ROUND(SUM(total_venta), 2) AS venta_total
# FROM ventas_temp GROUP BY tienda, categoria ORDER BY venta_total DESC LIMIT 10


## 🧪 Ejercicio propuesto

1. Crea un nuevo widget de tipo `combobox` para filtrar por **categoría**.
2. Calcula el **top 3 de categorías** por venta total usando `groupBy` + `agg` + `orderBy`.
3. Únete con un compañero en el mismo notebook (colaboración en vivo) y agreguen un comentario en una celda explicando el hallazgo más interesante.
4. Guarden el resultado como una nueva tabla `clase3_top_categorias`.

---
### Cierre del laboratorio
Ya sabes: adjuntar un notebook a un cluster, explorar su configuración, usar magic commands, transformar datos a escala con PySpark y compartir resultados con tu equipo mediante tablas y widgets.

**Siguiente notebook →** `02_mlflow_tracking_registry.ipynb`: vamos a entrenar un modelo y registrar todo el experimento con MLflow.
